# **EEG Schizophrenia Classification — Image Generation + EfficientNet (2026)**
### Pipeline: EEG CSV → Mel Spectrogram Images (HC-2000, SZ-2000) → EfficientNetB0

## Step 1 — Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no display needed
import matplotlib.pyplot as plt
import librosa
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# GPU memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

2026-05-13 04:35:06.413067: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778646906.914097      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778646907.058952      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778646908.165481      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778646908.165578      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778646908.165582      57 computation_placer.cc:177] computation placer alr

TF version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    print("Dirs:", dirs[:5])
    print("Files:", files[:5])
    print("-"*50)

/kaggle/input
Dirs: ['datasets']
Files: []
--------------------------------------------------
/kaggle/input/datasets
Dirs: ['broach']
Files: []
--------------------------------------------------
/kaggle/input/datasets/broach
Dirs: ['button-tone-sz']
Files: []
--------------------------------------------------
/kaggle/input/datasets/broach/button-tone-sz
Dirs: ['18.csv', '20.csv', '71.csv', '74.csv', '1.csv']
Files: ['columnLabels.csv', 'ERPdata.csv', 'demographic.csv', 'time.csv', 'mergedTrialData.csv']
--------------------------------------------------
/kaggle/input/datasets/broach/button-tone-sz/18.csv
Dirs: []
Files: ['18.csv']
--------------------------------------------------
/kaggle/input/datasets/broach/button-tone-sz/20.csv
Dirs: []
Files: ['20.csv']
--------------------------------------------------
/kaggle/input/datasets/broach/button-tone-sz/71.csv
Dirs: []
Files: ['71.csv']
--------------------------------------------------
/kaggle/input/datasets/broach/button-tone-sz/74.cs

## Step 2 — Subject Label Mapping
From `demographic.csv` output (confirmed from the run):
- **HC (Healthy):** subjects 0–23 and 58–65  
- **SZ (Schizophrenia):** subjects 24–57 and 66–80

In [4]:
from pathlib import Path
import pandas as pd

# ── Correct Dataset Path ─────────────────────────────────────
DATASET = Path('/kaggle/input/datasets/broach/button-tone-sz')

OUT_DIR = Path('/kaggle/working/eeg_images')
OUT_DIR.mkdir(exist_ok=True)

# ── Load demographic.csv ────────────────────────────────────
demographic = pd.read_csv(DATASET / 'demographic.csv')

print(demographic.head())

# column name contains leading space
print(demographic.columns)

# ── Subject → Label Mapping ─────────────────────────────────
# 0 = HC
# 1 = SZ

diagnosis_dict = dict(
    zip(
        demographic['subject'],
        demographic[' group']
    )
)

# ── Verify split ────────────────────────────────────────────
hc_subjects = [s for s, g in diagnosis_dict.items() if g == 0]
sz_subjects = [s for s, g in diagnosis_dict.items() if g == 1]

print(f'HC subjects ({len(hc_subjects)}):')
print(sorted(hc_subjects))

print(f'\nSZ subjects ({len(sz_subjects)}):')
print(sorted(sz_subjects))

   subject   group  gender   age   education
0        1       0       M    44        16.0
1        2       0       M    39        17.0
2        3       0       M    53        18.0
3        4       0       M    52        15.0
4        5       0       M    41        16.0
Index(['subject', ' group', ' gender', ' age', ' education'], dtype='object')
HC subjects (32):
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 59, 60, 61, 62, 63, 64, 65, 66]

SZ subjects (49):
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81]


## Step 3 — EEG → Mel Spectrogram Image (your code, wrapped into a function)

In [7]:
# ── Column schema ───────────────────────────────────────────────────────────
CHANNEL_NAMES = [
    'trial','sample','id','time',
    'Fp1','F7','T3','T5','O1',
    'Fp2','F8','T4','T6','O2',
    'F3','C3','P3',
    'F4','C4','P4'
]

# ── Montage: 4 bipolar chains ────────────────────────────────────────────────
MONTAGE_NAMES = ['LL', 'LP', 'RP', 'RR']
MONTAGE_FEATS = [
    ['Fp1','F7','T3','T5','O1'],   # LL
    ['Fp1','F3','C3','P3','O1'],   # LP
    ['Fp2','F4','C4','P4','O2'],   # RP
    ['Fp2','F8','T4','T6','O2'],   # RR
]

IMG_H, IMG_W = 128, 256   # spectrogram shape per panel
SR           = 200         # EEG sampling rate (Hz)

# safer + faster FFT settings
N_FFT        = 256
N_MELS       = 128

# better EEG frequency coverage
FMIN, FMAX   = 0, 40

WIN_LENGTH   = 64


def trial_to_spectrogram_image(trial_df, save_path):
    """
    Convert one EEG trial (DataFrame rows for that trial) into a
    2×2 Mel-spectrogram image and save as PNG.

    Returns True on success, False if any channel data is missing.
    """

    fig, axes = plt.subplots(2, 2, figsize=(8, 6))

    for k, (name, cols) in enumerate(zip(MONTAGE_NAMES, MONTAGE_FEATS)):

        # Check all required columns exist
        if not all(c in trial_df.columns for c in cols):
            plt.close(fig)
            return False

        img = np.zeros((IMG_H, IMG_W), dtype=np.float32)

        for i in range(len(cols) - 1):

            x = (
                trial_df[cols[i]].values
                -
                trial_df[cols[i + 1]].values
            )

            x = np.nan_to_num(
                x.astype(np.float32)
            )

            hop = max(1, len(x) // IMG_W)

            mel = librosa.feature.melspectrogram(
                y=x,
                sr=SR,
                hop_length=hop,
                n_fft=N_FFT,
                n_mels=N_MELS,
                fmin=FMIN,
                fmax=FMAX,
                win_length=WIN_LENGTH
            )

            mel = librosa.power_to_db(
                mel,
                ref=np.max
            )

            # Crop / pad to fixed shape
            mel = mel[:IMG_H, :IMG_W]

            if mel.shape[1] < IMG_W:

                mel = np.pad(
                    mel,
                    (
                        (0, 0),
                        (0, IMG_W - mel.shape[1])
                    ),
                    mode='constant'
                )

            img += mel

        # correct averaging
        img /= (len(cols) - 1)

        ax = axes.flatten()[k]

        ax.imshow(
            img,
            aspect='auto',
            origin='lower',
            cmap='viridis'
        )

        # removed title because CNN should not learn text
        # ax.set_title(f'EEG {name}')

        ax.axis('off')

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=100,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close(fig)

    return True


print('Image generation function ready.')

Image generation function ready.


## Step 4 — Generate 2000 HC + 2000 SZ Images

In [9]:
IMAGES_PER_CLASS = 2000

# Create output folders
for cls in ['HC', 'SZ']:
    Path(f'{OUT_DIR}/{cls}').mkdir(
        parents=True,
        exist_ok=True
    )


def get_csv_path(subject_num):
    """
    Return full path to subject CSV.
    """

    csv_path = f'{PART1}/{subject_num}.csv/{subject_num}.csv'

    if os.path.exists(csv_path):
        return csv_path

    return None


def generate_images_for_subjects(
    subject_list,
    label_str,
    max_images
):
    """
    Iterate over subjects, extract trials,
    generate spectrograms.

    Stops once max_images are saved.
    """

    saved = 0

    pbar = tqdm(
        total=max_images,
        desc=f'Generating {label_str}'
    )

    for subj in subject_list:

        if saved >= max_images:
            break

        csv_path = get_csv_path(subj)

        if csv_path is None:
            print(f'[WARN] Subject {subj} CSV not found — skipping')
            continue

        # Read CSV safely
        try:

            df = pd.read_csv(
                csv_path,
                header=None
            )

        except Exception as e:

            print(f'[ERROR] Could not read subject {subj}: {e}')
            continue

        # Handle extra columns safely
        extra = [
            f'extra_{i}'
            for i in range(
                df.shape[1] - len(CHANNEL_NAMES)
            )
        ]

        df.columns = CHANNEL_NAMES + extra

        # Process all trials
        for trial_num in df['trial'].unique():

            if saved >= max_images:
                break

            # Faster selection
            trial_df = df.loc[
                df['trial'] == trial_num
            ]

            # Skip very short / corrupted trials
            if len(trial_df) < 8000:
                continue

            save_path = (
                f'{OUT_DIR}/{label_str}/'
                f'subj{subj:03d}_'
                f'trial{int(trial_num):04d}.png'
            )

            try:

                ok = trial_to_spectrogram_image(
                    trial_df,
                    save_path
                )

                if ok:

                    saved += 1
                    pbar.update(1)

            except Exception as e:

                print(
                    f'[ERROR] Subject {subj} '
                    f'Trial {trial_num}: {e}'
                )

                continue

    pbar.close()

    print(f'\n→ {label_str}: {saved} images saved')

    return saved


# ── Generate HC images ──────────────────────────────────────
n_hc = generate_images_for_subjects(
    sorted(hc_subjects),
    'HC',
    IMAGES_PER_CLASS
)

# ── Generate SZ images ──────────────────────────────────────
n_sz = generate_images_for_subjects(
    sorted(sz_subjects),
    'SZ',
    IMAGES_PER_CLASS
)

# ── Final Summary ───────────────────────────────────────────
print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'HC Images : {n_hc}')
print(f'SZ Images : {n_sz}')
print(f'Total     : {n_hc + n_sz}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━')

print(f'\nImages saved to:\n{OUT_DIR}')

Generating HC:   0%|          | 0/2000 [00:00<?, ?it/s]

[WARN] Subject 1 CSV not found — skipping
[WARN] Subject 2 CSV not found — skipping
[WARN] Subject 3 CSV not found — skipping
[WARN] Subject 4 CSV not found — skipping
[WARN] Subject 5 CSV not found — skipping
[WARN] Subject 6 CSV not found — skipping
[WARN] Subject 7 CSV not found — skipping
[WARN] Subject 8 CSV not found — skipping
[WARN] Subject 9 CSV not found — skipping
[WARN] Subject 10 CSV not found — skipping
[WARN] Subject 11 CSV not found — skipping
[WARN] Subject 12 CSV not found — skipping
[WARN] Subject 13 CSV not found — skipping
[WARN] Subject 14 CSV not found — skipping
[WARN] Subject 15 CSV not found — skipping
[WARN] Subject 16 CSV not found — skipping
[WARN] Subject 17 CSV not found — skipping
[WARN] Subject 18 CSV not found — skipping
[WARN] Subject 19 CSV not found — skipping
[WARN] Subject 20 CSV not found — skipping
[WARN] Subject 21 CSV not found — skipping
[WARN] Subject 22 CSV not found — skipping
[WARN] Subject 23 CSV not found — skipping
[WARN] Subject 24 CS

Generating SZ:   0%|          | 0/2000 [00:00<?, ?it/s]

[WARN] Subject 25 CSV not found — skipping
[WARN] Subject 26 CSV not found — skipping
[WARN] Subject 27 CSV not found — skipping
[WARN] Subject 28 CSV not found — skipping
[WARN] Subject 29 CSV not found — skipping
[WARN] Subject 30 CSV not found — skipping
[WARN] Subject 31 CSV not found — skipping
[WARN] Subject 32 CSV not found — skipping
[WARN] Subject 33 CSV not found — skipping
[WARN] Subject 34 CSV not found — skipping
[WARN] Subject 35 CSV not found — skipping
[WARN] Subject 36 CSV not found — skipping
[WARN] Subject 37 CSV not found — skipping
[WARN] Subject 38 CSV not found — skipping
[WARN] Subject 39 CSV not found — skipping
[WARN] Subject 40 CSV not found — skipping
[WARN] Subject 41 CSV not found — skipping
[WARN] Subject 42 CSV not found — skipping
[WARN] Subject 43 CSV not found — skipping
[WARN] Subject 44 CSV not found — skipping
[WARN] Subject 45 CSV not found — skipping
[WARN] Subject 46 CSV not found — skipping
[WARN] Subject 47 CSV not found — skipping
[WARN] Subj

## Step 5 — Preview Sample Images

In [ ]:
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, cls in enumerate(['HC', 'SZ']):
    files = sorted(os.listdir(f'{OUT_DIR}/{cls}'))[:3]
    for col, fname in enumerate(files):
        img = Image.open(f'{OUT_DIR}/{cls}/{fname}')
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{cls} — {fname}', fontsize=9)
        axes[row, col].axis('off')
plt.suptitle('Sample Generated Spectrogram Images', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6 — Data Pipeline (Train / Val / Test Split)

In [ ]:
import shutil, random
from pathlib import Path

SPLIT_DIR  = '/kaggle/working/split'
IMG_SIZE   = (224, 224)   # EfficientNetB0 native input
BATCH_SIZE = 32
SEED       = 42
random.seed(SEED)

# Train 70% | Val 15% | Test 15%
SPLITS = {'train': 0.70, 'val': 0.15, 'test': 0.15}

for cls in ['HC', 'SZ']:
    all_files = sorted(Path(f'{OUT_DIR}/{cls}').glob('*.png'))
    random.shuffle(all_files)
    n = len(all_files)

    n_train = int(n * SPLITS['train'])
    n_val   = int(n * SPLITS['val'])

    split_files = {
        'train': all_files[:n_train],
        'val'  : all_files[n_train : n_train + n_val],
        'test' : all_files[n_train + n_val:]
    }

    for split, files in split_files.items():
        dest = Path(f'{SPLIT_DIR}/{split}/{cls}')
        dest.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy(f, dest / f.name)

    print(f'{cls}: train={len(split_files["train"])}, '
          f'val={len(split_files["val"])}, test={len(split_files["test"])}')

print('\nSplit complete.')

In [ ]:
# ── ImageDataGenerators ─────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=False,   # spectrograms are time-ordered — no H-flip
    zoom_range=0.05
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{SPLIT_DIR}/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    f'{SPLIT_DIR}/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    f'{SPLIT_DIR}/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)   # HC=0, SZ=1

## Step 7 — Build EfficientNetB0 Model

In [ ]:
def build_efficientnet_model(img_size=(224, 224, 3), fine_tune_from=None):
    """
    EfficientNetB0 with custom classification head.
    fine_tune_from: int — unfreeze layers from this index onward.
                    None = freeze all base layers (feature extraction only).
    """
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=img_size
    )

    if fine_tune_from is None:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:fine_tune_from]:
            layer.trainable = False

    inputs = keras.Input(shape=img_size)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    return model


model = build_efficientnet_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

## Step 8 — Phase 1: Feature Extraction (Frozen Base)

In [ ]:
callbacks_phase1 = [
    ModelCheckpoint(
        '/kaggle/working/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
    )
]

history1 = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks_phase1
)

print('Phase 1 done.')

## Step 9 — Phase 2: Fine-Tuning (Unfreeze Top Layers)

In [ ]:
# Unfreeze from layer 100 onward (top ~100 layers of EfficientNetB0)
base_model = model.layers[1]   # the EfficientNetB0 sub-model
base_model.trainable = True
FINE_TUNE_FROM = 100
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

# Recompile with a much lower LR for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    ModelCheckpoint(
        '/kaggle/working/best_phase2.keras',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=7, min_lr=1e-8, verbose=1
    )
]

history2 = model.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=callbacks_phase2
)

print('Phase 2 fine-tuning done.')

## Step 10 — Training Curves

In [ ]:
def plot_history(h1, h2):
    acc  = h1.history['accuracy']        + h2.history['accuracy']
    vacc = h1.history['val_accuracy']    + h2.history['val_accuracy']
    loss = h1.history['loss']            + h2.history['loss']
    vloss= h1.history['val_loss']        + h2.history['val_loss']
    ep1  = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(acc,  label='Train Acc')
    axes[0].plot(vacc, label='Val Acc')
    axes[0].axvline(ep1, color='gray', linestyle='--', label='Fine-tune start')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(loss,  label='Train Loss')
    axes[1].plot(vloss, label='Val Loss')
    axes[1].axvline(ep1, color='gray', linestyle='--', label='Fine-tune start')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

## Step 11 — Evaluate on Test Set

In [ ]:
# ── Test accuracy / loss ────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')

# ── Predictions ─────────────────────────────────────────────────────────────
y_pred_prob = model.predict(test_gen).ravel()
y_pred      = (y_pred_prob >= 0.5).astype(int)
y_true      = test_gen.classes

# Class index: HC=0, SZ=1 (alphabetical order)
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['HC', 'SZ']))

## Step 12 — Confusion Matrix + Sensitivity & Specificity

In [ ]:
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity  = TN / (TN + FP)
precision    = TP / (TP + FP)
f1           = 2 * precision * sensitivity / (precision + sensitivity)

print(f'Confusion Matrix:')
print(f'  TN (HC correct)  : {TN}')
print(f'  FP (HC→SZ wrong) : {FP}')
print(f'  FN (SZ→HC wrong) : {FN}')
print(f'  TP (SZ correct)  : {TP}')
print()
print(f'Sensitivity (Recall for SZ) : {sensitivity:.4f}')
print(f'Specificity (Recall for HC) : {specificity:.4f}')
print(f'Precision                   : {precision:.4f}')
print(f'F1-Score                    : {f1:.4f}')

# ── Plot confusion matrix ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['HC (pred)', 'SZ (pred)'],
    yticklabels=['HC (true)', 'SZ (true)'],
    ax=ax
)
ax.set_title('Confusion Matrix — EfficientNetB0')
plt.tight_layout()
plt.show()

## Step 13 — Save Final Model

In [ ]:
model.save('/kaggle/working/efficientnet_eeg_schizophrenia.keras')
print('Model saved to /kaggle/working/efficientnet_eeg_schizophrenia.keras')